# INVESTIGACIÓN 1 — ¿Cómo se enlaza central_product ↔ product?

## Setup (igual que antes, con el helper)

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

load_dotenv()
url = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME_ORIGEN')}"
)
engine = create_engine(url)

def q(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

OUT_DIR = Path("../docs/findings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK ✓")

Setup OK ✓


##  Inspeccionar contenido de las dos tablas

In [2]:
print("=== central_product (49 filas) ===")
cp = q("SELECT * FROM central_product ORDER BY product_id LIMIT 10;")
print(cp)
print(f"\nColumnas: {list(cp.columns)}")

=== central_product (49 filas) ===
   product_id                            name  category_id  brand_id     sku barcode  unit_cost  unit_price
0           1             Tensiómetro digital            1         1   SKU-1    BC-1      35.99       59.99
1           2               Oxímetro de pulso            1         2   SKU-2    BC-2      17.99       29.99
2           3           Termómetro infrarrojo            1         3   SKU-3    BC-3      14.94       24.90
3           4             Báscula inteligente            2         4   SKU-4    BC-4      20.99       34.99
4           5              Monitor de glucosa            1         5   SKU-5    BC-5      47.99       79.99
5           6           Cepillo dental sónico            5         6   SKU-6    BC-6      29.94       49.90
6           7             Purificador de aire            2         7   SKU-7    BC-7      89.99      149.99
7           8        Lámpara UV desinfectante            5         4   SKU-8    BC-8      23.94      

In [3]:
print("=== product (50 filas) ===")
p = q("SELECT * FROM product ORDER BY product_id LIMIT 10;")
print(p)
print(f"\nColumnas: {list(p.columns)}")

=== product (50 filas) ===
   product_id                            name     category manufacturer   price                 created_at
0           1             Tensiómetro digital  Diagnóstico        Omron   59.99 2026-04-04 11:23:48.591695
1           2               Oxímetro de pulso  Diagnóstico       Beurer   29.99 2026-04-04 11:23:48.591695
2           3           Termómetro infrarrojo  Diagnóstico     Medisana   24.90 2026-04-04 11:23:48.591695
3           4             Báscula inteligente     Wellness       Xiaomi   34.99 2026-04-04 11:23:48.591695
4           5              Monitor de glucosa  Diagnóstico       Abbott   79.99 2026-04-04 11:23:48.591695
5           6           Cepillo dental sónico      Higiene       Oral-B   49.90 2026-04-04 11:23:48.591695
6           7             Purificador de aire     Wellness      Philips  149.99 2026-04-04 11:23:48.591695
7           8        Lámpara UV desinfectante      Higiene       Xiaomi   39.90 2026-04-04 11:23:48.591695
8         

## Verificar candidatos de enlace

In [4]:
# Hipótesis 1: ¿Coinciden los product_id?
ids_central = set(q("SELECT product_id FROM central_product;")['product_id'])
ids_product = set(q("SELECT product_id FROM product;")['product_id'])

print(f"product_ids en central_product: {len(ids_central)}")
print(f"product_ids en product: {len(ids_product)}")
print(f"Intersección: {len(ids_central & ids_product)}")
print(f"En product pero NO en central: {sorted(ids_product - ids_central)}")
print(f"En central pero NO en product: {sorted(ids_central - ids_product)}")

product_ids en central_product: 49
product_ids en product: 50
Intersección: 49
En product pero NO en central: [29]
En central pero NO en product: []


In [5]:
# Hipótesis 2: ¿Coinciden los nombres?
nombres_match = q("""
    SELECT 
        cp.product_id AS cp_id,
        p.product_id AS p_id,
        cp.name AS cp_name,
        p.name AS p_name,
        cp.unit_cost,
        cp.unit_price AS cp_price,
        p.price AS p_price
    FROM central_product cp
    INNER JOIN product p ON LOWER(TRIM(cp.name)) = LOWER(TRIM(p.name))
    ORDER BY cp.product_id;
""")
print(f"Productos enlazados por NOMBRE: {len(nombres_match)} de 49 esperados")
nombres_match.head(10)

Productos enlazados por NOMBRE: 49 de 49 esperados


,cp_id,p_id,cp_name,p_name,unit_cost,cp_price,p_price
0,1,1,Tensiómetro digital,Tensiómetro digital,35.99,59.99,59.99
1,2,2,Oxímetro de pulso,Oxímetro de pulso,17.99,29.99,29.99
2,3,3,Termómetro infrarrojo,Termómetro infrarrojo,14.94,24.90,24.90
3,4,4,Báscula inteligente,Báscula inteligente,20.99,34.99,34.99
4,5,5,Monitor de glucosa,Monitor de glucosa,47.99,79.99,79.99
5,6,6,Cepillo dental sónico,Cepillo dental sónico,29.94,49.90,49.90
6,7,7,Purificador de aire,Purificador de aire,89.99,149.99,149.99
7,8,8,Lámpara UV desinfectante,Lámpara UV desinfectante,23.94,39.90,39.90
8,9,9,Nebulizador ultrasónico,Nebulizador ultrasónico,41.99,69.99,69.99
9,10,10,Tiras reactivas glucosa (pack),Tiras reactivas glucosa (pack),11.99,19.99,19.99


In [6]:
# Hipótesis 3: ¿Coinciden por SKU? (central_product tiene sku, product no parece tenerlo)
# Verificamos columnas
sku_check = q("""
    SELECT column_name, table_name 
    FROM information_schema.columns 
    WHERE table_schema='public' 
      AND column_name ILIKE '%sku%' OR column_name ILIKE '%barcode%' OR column_name ILIKE '%code%'
    ORDER BY table_name;
""")
print("Columnas con 'sku', 'barcode' o 'code':")
sku_check

Columnas con 'sku', 'barcode' o 'code':


,column_name,table_name
0,sku,central_product
1,barcode,central_product
2,city_code,city_zone
3,postal_code,city_zone
4,oprcode,pg_operator
5,postal_code,store
6,postal_code,warehouse
7,bin_code,warehouse_location


## Identificar el producto huérfano

In [7]:
# El producto que está en `product` (50) pero no en `central_product` (49)
# Mostrarlo para entender qué pasa
huerfano = q("""
    SELECT p.* 
    FROM product p
    LEFT JOIN central_product cp ON LOWER(TRIM(p.name)) = LOWER(TRIM(cp.name))
    WHERE cp.product_id IS NULL;
""")
print("Producto(s) en `product` SIN match en `central_product`:")
huerfano

Producto(s) en `product` SIN match en `central_product`:


,product_id,name,category,manufacturer,price,created_at
0,29,Sensor temperatura inteligente,Domótica Salud,Xiaomi,19.99,2026-04-04 11:23:48.591695


In [8]:
# ¿Y al revés? Productos en central que no estén en product
huerfano_inverso = q("""
    SELECT cp.* 
    FROM central_product cp
    LEFT JOIN product p ON LOWER(TRIM(cp.name)) = LOWER(TRIM(p.name))
    WHERE p.product_id IS NULL;
""")
print("Producto(s) en `central_product` SIN match en `product`:")
huerfano_inverso

Producto(s) en `central_product` SIN match en `product`:


,product_id,name,category_id,brand_id,sku,barcode,unit_cost,unit_price


In [9]:
# ¿El producto huérfano se ha vendido alguna vez?
huerfano_id = huerfano['product_id'].tolist()
if huerfano_id:
    ventas_huerfano = q(f"""
        SELECT COUNT(*) AS num_ventas, SUM(quantity) AS unidades
        FROM sale_item
        WHERE product_id IN ({','.join(map(str, huerfano_id))});
    """)
    print(f"Ventas del/los producto(s) huérfano(s):")
    print(ventas_huerfano)

Ventas del/los producto(s) huérfano(s):
   num_ventas  unidades
0         711      1426


#  INVESTIGACIÓN 2 — ¿Cómo se enlaza city_zone?

In [10]:
print("=== city_zone (42 filas) ===")
cz = q("SELECT * FROM city_zone LIMIT 10;")
cz

=== city_zone (42 filas) ===


,postal_code,district,area_type,zone_orientation,city_code,city
0,28030,Moratalaz,Periférica,NaN,28,Madrid
1,28001,Salamanca,Céntrica,Centro,28,Madrid
2,28004,Centro,Céntrica,Centro,28,Madrid
3,28005,Centro,Céntrica,Centro,28,Madrid
4,28012,Centro,Céntrica,Centro,28,Madrid
5,28013,Centro,Céntrica,Centro,28,Madrid
6,28014,Centro,Céntrica,Centro,28,Madrid
7,28009,Retiro,Céntrica,Centro,28,Madrid
8,28006,Salamanca,Céntrica,Centro,28,Madrid
9,28010,Chamberí,Céntrica,Centro,28,Madrid


In [11]:
# Hipótesis: city_zone.postal_code enlaza con customer.postal_code o store.postal_code
# Pero customer no tiene postal_code... vamos a comprobarlo

print("Columnas de customer y store:")
cols_check = q("""
    SELECT table_name, column_name 
    FROM information_schema.columns
    WHERE table_schema='public' AND table_name IN ('customer','store')
    ORDER BY table_name, ordinal_position;
""")
cols_check

Columnas de customer y store:


,table_name,column_name
0,customer,customer_id
1,customer,first_name
2,customer,last_name
3,customer,last_name2
4,customer,email
5,customer,phone
6,customer,created_at
7,store,store_id
8,store,name
9,store,address


In [12]:
# Si store tiene postal_code, probemos el match
match_store = q("""
    SELECT 
        s.store_id, s.name AS store_name, s.postal_code, s.city,
        cz.district, cz.area_type, cz.zone_orientation
    FROM store s
    LEFT JOIN city_zone cz ON s.postal_code = cz.postal_code
    ORDER BY s.store_id;
""")
print(f"Stores con info de city_zone: {match_store['district'].notna().sum()} de {len(match_store)}")
match_store.head(10)

Stores con info de city_zone: 20 de 20


,store_id,store_name,postal_code,city,district,area_type,zone_orientation
0,1,Store Retiro,28009,Madrid,Retiro,Céntrica,Centro
1,2,Store Salamanca,28001,Madrid,Salamanca,Céntrica,Centro
2,3,Store Chamartín,28016,Madrid,Chamartín,Céntrica,Norte
3,4,Store La Latina,28005,Madrid,Centro,Céntrica,Centro
4,5,Store Sol,28013,Madrid,Centro,Céntrica,Centro
5,6,Store Moncloa,28008,Madrid,Moncloa-Aravaca,Céntrica,Noroeste
6,7,Store Atocha,28012,Madrid,Centro,Céntrica,Centro
7,8,Store Vallecas,28018,Madrid,Puente de Vallecas,Periférica,Sureste
8,9,Store Usera,28026,Madrid,Usera,Periférica,Sur
9,10,Store Carabanchel,28019,Madrid,Carabanchel,Periférica,Suroeste


#  INVESTIGACIÓN 3 — Verificar return_item.reason_id ↔ return_reason

In [13]:
print("=== return_reason ===")
rr = q("SELECT * FROM return_reason;")
rr

=== return_reason ===


,reason_id,reason,active
0,1,No cumple expectativas,True
1,2,Producto defectuoso,True
2,3,Error en el pedido,True
3,4,Problema de funcionamiento,True
4,5,Cliente arrepentido,True
5,6,Desconocida,False


In [14]:
# Verificar si return_item.reason_id apunta a return_reason.reason_id
match_returns = q("""
    SELECT 
        ri.reason_id,
        rr.reason,
        COUNT(*) AS num_devoluciones
    FROM return_item ri
    LEFT JOIN return_reason rr ON ri.reason_id = rr.reason_id
    GROUP BY ri.reason_id, rr.reason
    ORDER BY num_devoluciones DESC;
""")
print(f"Devoluciones por motivo:")
match_returns

Devoluciones por motivo:


,reason_id,reason,num_devoluciones
0,2,Producto defectuoso,482
1,5,Cliente arrepentido,479
2,4,Problema de funcionamiento,475
3,1,No cumple expectativas,455
4,3,Error en el pedido,439


In [15]:
# ¿Hay reason_id en return_item que NO existan en return_reason?
orphans_returns = q("""
    SELECT DISTINCT ri.reason_id
    FROM return_item ri
    LEFT JOIN return_reason rr ON ri.reason_id = rr.reason_id
    WHERE rr.reason_id IS NULL AND ri.reason_id IS NOT NULL;
""")
print(f"reason_ids huérfanos en return_item: {len(orphans_returns)}")
orphans_returns

reason_ids huérfanos en return_item: 0


,reason_id


# VALIDACIÓN — Integridad real de las FKs declaradas

In [16]:
# Aunque haya FKs declaradas, comprobamos que no hay valores NULL inesperados o filas huérfanas
fk_checks = [
    ("sale", "customer_id", "customer", "customer_id"),
    ("sale", "store_id", "store", "store_id"),
    ("sale_item", "sale_id", "sale", "sale_id"),
    ("sale_item", "product_id", "product", "product_id"),
    ("sale_item", "offer_id", "offer", "offer_id"),
    ("return_item", "sale_item_id", "sale_item", "sale_item_id"),
    ("inventory", "store_id", "store", "store_id"),
    ("inventory", "product_id", "product", "product_id"),
]

resultados = []
for tabla_o, col_o, tabla_d, col_d in fk_checks:
    nulls = q(f"SELECT COUNT(*) AS n FROM {tabla_o} WHERE {col_o} IS NULL;")['n'][0]
    huerfanos = q(f"""
        SELECT COUNT(*) AS n
        FROM {tabla_o} o
        LEFT JOIN {tabla_d} d ON o.{col_o} = d.{col_d}
        WHERE o.{col_o} IS NOT NULL AND d.{col_d} IS NULL;
    """)['n'][0]
    resultados.append({
        'fk': f"{tabla_o}.{col_o} → {tabla_d}.{col_d}",
        'nulls': nulls,
        'huerfanos': huerfanos
    })

df_fk_check = pd.DataFrame(resultados)
df_fk_check.to_csv(OUT_DIR / "validacion_fks.csv", index=False)
df_fk_check

,fk,nulls,huerfanos
0,sale.customer_id → customer.customer_id,0,0
1,sale.store_id → store.store_id,0,0
2,sale_item.sale_id → sale.sale_id,0,0
3,sale_item.product_id → product.product_id,0,0
4,sale_item.offer_id → offer.offer_id,42547,0
5,return_item.sale_item_id → sale_item.sale_item_id,0,0
6,inventory.store_id → store.store_id,0,0
7,inventory.product_id → product.product_id,0,0


# VALIDACIÓN DE NEGOCIO — Coherencia de totales (CRÍTICO para CLTV)

## sale.total vs suma de sale_item.subtotal

In [17]:
coherencia_total = q("""
    WITH sumas AS (
        SELECT sale_id, SUM(subtotal) AS suma_items
        FROM sale_item
        GROUP BY sale_id
    )
    SELECT 
        s.sale_id,
        s.total AS total_sale,
        su.suma_items,
        ROUND((s.total - su.suma_items)::numeric, 2) AS diferencia
    FROM sale s
    LEFT JOIN sumas su ON s.sale_id = su.sale_id
    WHERE ABS(s.total - COALESCE(su.suma_items, 0)) > 0.01
    ORDER BY ABS(s.total - COALESCE(su.suma_items, 0)) DESC
    LIMIT 20;
""")
print(f"Ventas con incoherencia entre sale.total y suma de items: {len(coherencia_total)}")
coherencia_total

Ventas con incoherencia entre sale.total y suma de items: 1


,sale_id,total_sale,suma_items,diferencia
0,13009,324.77,321.77,3.0


In [18]:
# Estadísticas globales de la diferencia
stats_dif = q("""
    WITH sumas AS (
        SELECT sale_id, SUM(subtotal) AS suma_items
        FROM sale_item
        GROUP BY sale_id
    )
    SELECT 
        COUNT(*) AS total_ventas,
        COUNT(CASE WHEN ABS(s.total - COALESCE(su.suma_items, 0)) < 0.01 THEN 1 END) AS coherentes,
        COUNT(CASE WHEN ABS(s.total - COALESCE(su.suma_items, 0)) >= 0.01 THEN 1 END) AS incoherentes,
        ROUND(AVG(s.total - COALESCE(su.suma_items, 0))::numeric, 2) AS dif_media,
        ROUND(MAX(ABS(s.total - COALESCE(su.suma_items, 0)))::numeric, 2) AS dif_max_abs
    FROM sale s
    LEFT JOIN sumas su ON s.sale_id = su.sale_id;
""")
stats_dif

,total_ventas,coherentes,incoherentes,dif_media,dif_max_abs
0,20000,19999,1,0.0,3.0


## sale_item.subtotal vs quantity * unit_price

In [19]:
coherencia_subtotal = q("""
    SELECT 
        COUNT(*) AS total_items,
        COUNT(CASE WHEN ABS(subtotal - quantity * unit_price) < 0.01 THEN 1 END) AS calculados_directos,
        COUNT(CASE WHEN ABS(subtotal - quantity * unit_price) >= 0.01 THEN 1 END) AS con_descuento_o_diferencia,
        ROUND(AVG(quantity * unit_price - subtotal)::numeric, 4) AS descuento_medio
    FROM sale_item;
""")
coherencia_subtotal

,total_items,calculados_directos,con_descuento_o_diferencia,descuento_medio
0,42555,42547,8,0.0011


In [20]:
# Si hay items donde subtotal != quantity*unit_price, ¿es por la oferta?
descuento_oferta = q("""
    SELECT 
        CASE WHEN offer_id IS NULL THEN 'sin_oferta' ELSE 'con_oferta' END AS tipo,
        COUNT(*) AS items,
        COUNT(CASE WHEN ABS(subtotal - quantity * unit_price) >= 0.01 THEN 1 END) AS items_con_diferencia
    FROM sale_item
    GROUP BY tipo;
""")
descuento_oferta

,tipo,items,items_con_diferencia
0,con_oferta,8,8
1,sin_oferta,42547,0
